In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from ast import literal_eval
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity
from nltk.stem.snowball import SnowballStemmer
from nltk.stem.wordnet import WordNetLemmatizer
from nltk.corpus import wordnet


import warnings; warnings.simplefilter('ignore')

In [2]:
movies = pd.read_csv('/content/drive/MyDrive/LLM RAG Recommender - MovieLens/Data/ml-32m/movies_enriched.csv')

In [3]:
movies.head()

,movieId,tmdbId,title,overview,tagline,runtime,budget,revenue,release_date,genres,vote_average,vote_count,poster_path,director,cast,keywords,production_companies,collection_name,tmdb_recommendation_ids
0,1,862,Toy Story,"Led by Woody, Andy's toys live happily in his ...",The adventure takes off when toys come to life!,81,30000000,394436586,1995-11-22,"['Family', 'Comedy', 'Animation', 'Adventure']",7.970,19297,/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg,John Lasseter,"['Tom Hanks', 'Tim Allen', 'Don Rickles', 'Jim...","['rescue', 'friendship', 'mission', 'jealousy'...",['Pixar'],Toy Story Collection,"[863, 9487, 10193, 8587, 585]"
1,2,8844,Jumanji,When siblings Judy and Peter discover an encha...,It's a jungle in here.,104,65000000,262821940,1995-12-15,"['Adventure', 'Fantasy', 'Family']",7.242,10971,/iWV47r6kFneCiApgrMII5HSkfHw.jpg,Joe Johnston,"['Robin Williams', 'Kirsten Dunst', 'Bradley P...","['giant insect', 'board game', 'disappearance'...","['TriStar Pictures', 'Interscope Communication...",Jumanji Collection,"[353486, 6795, 788, 1593, 879]"
2,3,15602,Grumpier Old Men,A family wedding reignites the ancient feud be...,Still Yelling. Still Fighting. Still Ready for...,101,25000000,71500000,1995-12-22,"['Romance', 'Comedy']",6.500,409,/1FSXpj5e8l4KH6nVFO5SPUeraOt.jpg,Howard Deutch,"['Walter Matthau', 'Jack Lemmon', 'Ann-Margret...","['fishing', 'sequel', 'old man', 'best friend'...","['Lancaster Gate', 'Warner Bros. Pictures']",Grumpy Old Men Collection,"[11520, 39242, 24732, 30066, 50766]"
3,4,31357,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",Friends are the people who let you be yourself...,127,16000000,81452156,1995-12-22,"['Comedy', 'Drama', 'Romance']",6.260,179,/qJU6rfil5xLVb5HpJsmmfeSK254.jpg,Forest Whitaker,"['Whitney Houston', 'Angela Bassett', 'Loretta...","['based on novel or book', 'single mother', 'd...",['20th Century Fox'],NaN,"[31000, 21539, 16158, 33644, 64255]"
4,5,11862,Father of the Bride Part II,Just when George Banks has recovered from his ...,Just when his world is back to normal... he's ...,106,0,76594107,1995-12-08,"['Comedy', 'Family']",6.266,777,/rj4LBtwQ0uGrpBnCELr716Qo3mw.jpg,Charles Shyer,"['Steve Martin', 'Diane Keaton', 'Martin Short...","['daughter', 'baby', 'parent child relationshi...","['Touchstone Pictures', 'Sandollar Productions']",Father of the Bride (Steve Martin) Collection,"[11846, 65137, 22968, 14949, 13759]"


In [4]:
movies.shape

(86315, 19)

In [5]:
import numpy as np
from ast import literal_eval

# replacing nan with empty list []
movies['genres'] = movies['genres'].apply(lambda x: [] if x is np.nan else x)

# converting string into actual list which is the python object
movies['genres'] = movies['genres'].apply(lambda x: literal_eval(x) if isinstance(x, str) else x)

# Might remove later
movies['genres'] = movies['genres'].apply(lambda x: [i for i in x] if isinstance(x, list) else [])

In [6]:
movies['genres']

,genres
0,"[Family, Comedy, Animation, Adventure]"
1,"[Adventure, Fantasy, Family]"
2,"[Romance, Comedy]"
3,"[Comedy, Drama, Romance]"
4,"[Comedy, Family]"
...,...
86310,[Drama]
86311,"[Drama, Comedy]"
86312,[Drama]
86313,[Drama]


In [7]:
movies.head(2)

,movieId,tmdbId,title,overview,tagline,runtime,budget,revenue,release_date,genres,vote_average,vote_count,poster_path,director,cast,keywords,production_companies,collection_name,tmdb_recommendation_ids
0,1,862,Toy Story,"Led by Woody, Andy's toys live happily in his ...",The adventure takes off when toys come to life!,81,30000000,394436586,1995-11-22,"[Family, Comedy, Animation, Adventure]",7.970,19297,/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg,John Lasseter,"['Tom Hanks', 'Tim Allen', 'Don Rickles', 'Jim...","['rescue', 'friendship', 'mission', 'jealousy'...",['Pixar'],Toy Story Collection,"[863, 9487, 10193, 8587, 585]"
1,2,8844,Jumanji,When siblings Judy and Peter discover an encha...,It's a jungle in here.,104,65000000,262821940,1995-12-15,"[Adventure, Fantasy, Family]",7.242,10971,/iWV47r6kFneCiApgrMII5HSkfHw.jpg,Joe Johnston,"['Robin Williams', 'Kirsten Dunst', 'Bradley P...","['giant insect', 'board game', 'disappearance'...","['TriStar Pictures', 'Interscope Communication...",Jumanji Collection,"[353486, 6795, 788, 1593, 879]"


In [8]:
movies['overview'] = movies['overview'].fillna('')
movies['tagline'] = movies['tagline'].fillna('')
movies['description'] = movies['overview'] + movies['tagline']
movies['title'] = movies['title'].fillna('')
movies['director'] = movies['director'].fillna('')


movies['keywords'] = movies['keywords'].apply(lambda x: [] if x is np.nan else x)
movies['keywords'] = movies['keywords'].apply(lambda x: literal_eval(x) if isinstance(x, str) else x)
movies['keywords'] = movies['keywords'].apply(lambda x: [i for i in x] if isinstance(x, list) else [])

movies['cast'] = movies['cast'].apply(lambda x: [] if x is np.nan else x)
movies['cast'] = movies['cast'].apply(lambda x: literal_eval(x) if isinstance(x, str) else x)
movies['cast'] = movies['cast'].apply(lambda x: [i for i in x] if isinstance(x, list) else [])
movies['cast'] = movies['cast'].apply(lambda x: x[:3] if len(x) >=3 else x)

movies['production_companies'] = movies['production_companies'].apply(lambda x: [] if x is np.nan else x)
movies['production_companies'] = movies['production_companies'].apply(lambda x: literal_eval(x) if isinstance(x, str) else x)
movies['production_companies'] = movies['production_companies'].apply(lambda x: [i for i in x] if isinstance(x, list) else [])
movies['production_companies'] = movies['production_companies'].apply(lambda x: x[:3] if len(x) >=3 else x)


In [9]:
movies.cast, movies.production_companies

(0                      [Tom Hanks, Tim Allen, Don Rickles]
 1          [Robin Williams, Kirsten Dunst, Bradley Pierce]
 2               [Walter Matthau, Jack Lemmon, Ann-Margret]
 3        [Whitney Houston, Angela Bassett, Loretta Devine]
 4               [Steve Martin, Diane Keaton, Martin Short]
                                ...                        
 86310          [Damián Alcázar, Grapa Paola, María Zubiri]
 86311    [Siobhan Fallon Hogan, Peter Macon, Robert Pat...
 86312    [Taraneh Alidoosti, Mahtab Keramati, Masoud Ka...
 86313      [Jan Sterling, James MacArthur, William Windom]
 86314                            [Ueli Steck, Dani Arnold]
 Name: cast, Length: 86315, dtype: object,
 0                                                  [Pixar]
 1        [TriStar Pictures, Interscope Communications, ...
 2                  [Lancaster Gate, Warner Bros. Pictures]
 3                                       [20th Century Fox]
 4             [Touchstone Pictures, Sandollar Production

In [10]:
movies.description

,description
0,"Led by Woody, Andy's toys live happily in his ..."
1,When siblings Judy and Peter discover an encha...
2,A family wedding reignites the ancient feud be...
3,"Cheated on, mistreated and stepped on, the wom..."
4,Just when George Banks has recovered from his ...
...,...
86310,Ronnie Monroy has had an unmeaningful life as ...
86311,A death row prisoner with 10 days left to live...
86312,"Elham is a young, divorced Iranian woman. Seek..."
86313,"In Vietnam, aspiring actor Johnny Taylor is gi..."


In [11]:
movies['cast'] = movies['cast'].apply(lambda x: [str.lower(i.replace(" ", "")) for i in x])
movies['director'] = movies['director'].apply(lambda x: str.lower(x.replace(" ", "")))


In [12]:
def weighted_rating(x, m, C):
    v = x['vote_count']
    R = x['vote_average']
    return (v/(v+m) * R) + (m/(m+v) * C)

In [14]:
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=1)

In [15]:
tfidf_matrix = tfidf.fit_transform(movies['description'])

In [16]:
tfidf_matrix.shape

(86315, 1736558)

In [17]:
from ast import literal_eval
from sklearn.feature_extraction.text import CountVectorizer

In [18]:
def create_soup(x):
    return (
        ' '.join(x['keywords']) + ' ' +
        ' '.join(x['cast']) + ' ' +
        x['director'] + ' ' +
        ' '.join(x['genres']) + ' ' +
        ' '.join(x['production_companies'])
    )

movies['soup'] = movies.apply(create_soup, axis=1)

In [20]:
count = CountVectorizer(analyzer='word', ngram_range=(1, 1), min_df=1, stop_words='english')
count_matrix = count.fit_transform(movies['soup'])

In [21]:
count_matrix

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 1332414 stored elements and shape (86315, 162458)>

In [22]:
from scipy.sparse import hstack

# Stack the two matrices horizontally
combined_matrix = hstack([tfidf_matrix, count_matrix])

In [23]:
combined_matrix.shape

(86315, 1899016)

In [25]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


movies = movies.reset_index()
indices = pd.Series(movies.index, index=movies['title'])

In [44]:
def get_recommendations_batch(titles, combined_matrix, movies_df, indices, batch_size=10, top_n=10, candidate_pool=50):

    results = {}

    # Process in batches
    for i in range(0, len(titles), batch_size):
        batch_titles = titles[i : i + batch_size]

        # Get valid indices
        batch_indices = []
        valid_titles = []

        for t in batch_titles:
            if t in indices:
                idx = indices[t]
                # Handle duplicate titles
                if isinstance(idx, (pd.Series, pd.Index, np.ndarray, list)):
                    idx = idx.iloc[0] if hasattr(idx, 'iloc') else idx[0]

                batch_indices.append(idx)
                valid_titles.append(t)
            else:
                print(f"Warning: '{t}' not found in indices.")

        if not batch_indices:
            continue

        # Extract feature vectors
        target_vectors = combined_matrix[batch_indices]

        # Compute cosine similarity
        batch_sim_scores = cosine_similarity(target_vectors, combined_matrix)

        # Process each movie
        for k, title in enumerate(valid_titles):
            scores = batch_sim_scores[k]

            top_indices = np.argpartition(scores, -(candidate_pool + 1))[-(candidate_pool + 1):]
            top_indices = top_indices[np.argsort(scores[top_indices])[::-1]]

            # Exclude input movie
            final_indices = [idx for idx in top_indices if idx != batch_indices[k]][:candidate_pool]

            # Get metadata
            recs = movies_df.iloc[final_indices][['title', 'vote_count', 'vote_average', 'year']].copy()
            recs['similarity'] = scores[final_indices]

            # Filter for Quality

            C = recs['vote_average'].mean()
            m = recs['vote_count'].quantile(0.20)
            recs = recs[recs['vote_count'] >= m]

            # Calculate Score
            recs['wr_score'] = recs.apply(lambda x: weighted_rating(x, m, C), axis=1)


            results[title] = recs.sort_values('similarity', ascending=False).head(top_n)

    return results

In [46]:
import time


test_movies = [
    'Toy Story', 'Jumanji', 'Grumpier Old Men', 'Heat', 'Sabrina', 'Tom and Huck',
    'Sudden Death', 'GoldenEye', 'The American President', 'Dracula: Dead and Loving It'
]

start = time.time()
recs_1 = get_recommendations_batch(test_movies, combined_matrix, movies, indices, batch_size=1)
end = time.time()
print(f"Batch Size 1 took: {end - start:.4f} seconds")


recs_10 = get_recommendations_batch(test_movies, combined_matrix, movies, indices, batch_size=1000)



print(recs_10['Jumanji'])

Batch Size 1 took: 5.5198 seconds
                                                   title  vote_count  \
19908                     Percy Jackson: Sea of Monsters        5264   
14268  Percy Jackson & the Olympians: The Lightning T...        7669   
12291           The Chronicles of Narnia: Prince Caspian        6598   
14310                                Alice in Wonderland       14465   
15199                                Alice in Wonderland          62   
2057                               The NeverEnding Story        4219   
19254                          Oz the Great and Powerful        6675   
32323                                   Ready Player One       16372   
49525                     Jumanji: Welcome to the Jungle       14138   
57781                                      Chaos Walking        2459   

       vote_average  year  similarity  wr_score  
19908         6.025  2013    0.344829  6.027948  
14268         6.209  2010    0.330672  6.209574  
12291         6.628  20

In [49]:
import time
import random

# Selecting 100 Random Movies
random_titles = movies['title'].sample(n=100, random_state=42).tolist()

print(f"Selected {len(random_titles)} random movies for testing.")
print(f"Examples: {random_titles[:5]}")




test_results = get_recommendations_batch(
    random_titles,
    combined_matrix,
    movies,
    indices,
    batch_size=1000,
    top_n=50,
    candidate_pool=100
)


# looking at random results to see if they make sense
for i in range(10):
    title = random_titles[i]
    print(f"\nSource Movie: {title}")
    if title in test_results:
        print(test_results[title][['title', 'similarity', 'wr_score']].head(100))
    else:
        print("No recommendations generated (possibly filtered out).")

Selected 100 random movies for testing.
Examples: ['The Penthouse', 'Jinn', 'Robot on the Road', 'Pigs 2: The Last Blood', 'Crooklyn']

Source Movie: The Penthouse
                            title  similarity  wr_score
52300            The Lost Brother    0.410391  6.778114
51611             Dead Slow Ahead    0.389490  6.482203
51579                 Mrs. Harris    0.382692  5.704024
79083            The Night Doctor    0.381771  6.445280
23937             Penthouse North    0.377815  5.710899
48371               The Next Skin    0.374634  5.696419
23614                  On the Ice    0.374634  6.108066
23992        The Body of My Enemy    0.368606  6.324577
39647         A Friend of Vincent    0.368563  5.787305
24059                       Prowl    0.368421  5.030127
83654                    Scorpion    0.368421  5.446472
67927                  Dirt Music    0.368421  6.138481
68112        The Crimes That Bind    0.367884  6.397501
15068              8th Wonderland    0.367884  6.051